# **Downloading the required Libraries**

In [ ]:
!pip install -q sentence-transformers transformers torch pandas numpy scipy scikit-learn matplotlib seaborn reportlab accelerate rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.5 MB/s eta 0:00:00


# **Importing the required libraries**

In [ ]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup
import re
from sklearn.model_selection import train_test_split
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
import nltk
import ssl
import rank_bm25

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from typing import List, Dict, Tuple, Optional
from sentence_transformers import CrossEncoder, util
from transformers import pipeline
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm
import gc
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)


from reportlab.lib.pagesizes import A4, letter
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, PageBreak, KeepTogether
)
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.pdfgen import canvas
import matplotlib
matplotlib.use('Agg')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.




---

#  **Gemini Dataset Training**


---



In [ ]:
with open("/content/gemini-sample.json", "r") as f:
    print(f.read()[:1000])


[
  {
    "id": "conv_001",
    "domain": "Programming",
    "topic": "React Hook Troubleshooting",
    "turns": [
      {"role": "user", "content": "I'm trying to use `useEffect` in React to fetch data, but it keeps looping infinitely."},
      {"role": "assistant", "content": "That usually happens when the dependency array is missing or configured incorrectly. Can you show me your `useEffect` code snippet?"},
      {"role": "user", "content": "Sure, here it is:\n```javascript\nuseEffect(() => {\n  fetchData();\n});\n```"},
      {"role": "assistant", "content": "I see the issue. You haven't passed a dependency array as the second argument. Without it, `useEffect` runs after *every* render. Since setting state triggers a re-render, it creates an infinite loop. You should add an empty array `[]` if you only want it to run once on mount."},
      {"role": "user", "content": "Oh, right. Like this?\n```javascript\nuseEffect(() => {\n  fetchData();\n}, []);\n```\nBut what if `fetchData` de

In [ ]:
df = pd.read_json("/content/gemini-sample.json")
print(df.head())

         id       domain                       topic  \
0  conv_001  Programming  React Hook Troubleshooting   
1  conv_002      Cooking    Substituting Ingredients   
2  conv_003       Career          Salary Negotiation   
3  conv_004   Technology          Excel Formula Help   
4  conv_005       Health           Marathon Training   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
with open("/content/gemini-sample.json", "r") as f:
    gemini_data = json.load(f)

# Flatten conversations
df_gemini = pd.json_normalize(
    gemini_data,
    record_path="turns",
    meta=["id", "topic"]
)


df_gemini.to_csv("/content/gemini-sample.csv", index=False)

print("gemini-sample.csv exported")
print(df_gemini.head())


gemini-sample.csv exported
        role  \
0       user   
1  assistant   
2       user   
3  assistant   
4       user   

                                                                                                                                                                                                                                                                                 content  \
0                                                                                                                                                                                                 I'm trying to use `useEffect` in React to fetch data, but it keeps looping infinitely.   
1                                                                                                                                                    That usually happens when the dependency array is missing or configured incorrectly. Can you show me your `useEffect` code snippet?   
2                       

In [ ]:
df = pd.read_csv("gemini-sample.csv")


df = df.drop(columns=["topic"])

# Saving the updated CSV
df.to_csv("gemini-sample-updated.csv", index=False)


df.head()

,role,content,id
0,user,"I'm trying to use `useEffect` in React to fetch data, but it keeps looping infinitely.",conv_001
1,assistant,That usually happens when the dependency array is missing or configured incorrectly. Can you show me your `useEffect` code snippet?,conv_001
2,user,"Sure, here it is:\n```javascript\nuseEffect(() => {\n fetchData();\n});\n```",conv_001
3,assistant,"I see the issue. You haven't passed a dependency array as the second argument. Without it, `useEffect` runs after *every* render. Since setting state triggers a re-render, it creates an infinite loop. You should add an empty array `[]` if you only want it to run once on mount.",conv_001
4,user,"Oh, right. Like this?\n```javascript\nuseEffect(() => {\n fetchData();\n}, []);\n```\nBut what if `fetchData` depends on a `userId` prop?",conv_001




---

#  **Perplexity Dataset Training**


---



In [ ]:
with open("/content/perplexity-sample.json", "r") as f:
    print(f.read()[:1000])

[
  {
    "id": 1,
    "topic": "Planning a Trip to Japan",
    "turns": [
      {"role": "user", "message": "I'm planning a trip to Japan next spring. Where should I go?"},
      {"role": "assistant", "message": "Japan in spring is amazing for cherry blossoms! Start with Tokyo for modern vibes, then Kyoto for temples and history. How many days do you have?"},
      {"role": "user", "message": "About 10 days. I love food and hiking."},
      {"role": "assistant", "message": "Perfect. Days 1-3: Tokyo (Sushi at Tsukiji, Shibuya Crossing). Day 4: Train to Hakone for hiking Mt. Fuji views and onsen. Days 5-8: Kyoto (Fushimi Inari hike, kaiseki meals). Day 9: Nara for deer park. Day 10: Back to Tokyo. Budget?"},
      {"role": "user", "message": "Mid-range, around $2000 excluding flights. Any tips for cherry blossom spots?"},
      {"role": "assistant", "message": "That fits. Ueno Park in Tokyo and Philosopher's Path in Kyoto peak early April. Book ryokans early. Get a JR Pass for trains. V

In [ ]:
with open("/content/perplexity-sample.json", "r") as f:
    for i, line in enumerate(f, start=1):
        if i >= 12 and i <= 20:
            print(i, line)


12       {"role": "user", "message": "No allergies, but mostly vegetarian."},

13       {"role": "assistant", "message": "Tofu and veggie tempura abound. Try shojin ryori in Kyoto temples. Pack layers—spring weather varies."},

14       {"role": "user", "message": "Sounds great! App recommendations?"},

15       {"role": "assistant": "Hyperdia for trains, Google Translate for menus, Tabelog for food spots."}

16     ]

17   },

18   {

19     "id": 2,

20     "topic": "Learning Python Programming",



In [ ]:
with open("/content/perplexity-sample.json", "r") as f:
    for i, line in enumerate(f, start=1):
        if '"role": "assistant":' in line or '"role": "user":' in line:
            print(f"Invalid JSON at line {i}: {line.strip()}")

print('\n')

try:
    with open("/content/perplexity-sample.json", "r") as f:
        json.load(f)
    print("JSON is valid")
except json.JSONDecodeError as e:
    print("JSON is invalid")
    print(e)

print('\n')

with open("/content/perplexity-sample.json", "r") as f:
    text = f.read()

text = re.sub(
    r'\{"role":\s*"assistant":',
    '{"role": "assistant", "message":',
    text
)

text = re.sub(
    r'\{"role":\s*"user":',
    '{"role": "user", "message":',
    text
)

with open("/content/perplexity-sample-fixed.json", "w") as f:
    f.write(text)

print("Fixed file written: perplexity-sample-fixed.json")

print('\n')

with open("/content/perplexity-sample-fixed.json", "r") as f:
    json.load(f)

print("JSON is valid")


Invalid JSON at line 15: {"role": "assistant": "Hyperdia for trains, Google Translate for menus, Tabelog for food spots."}


JSON is invalid
Expecting ',' delimiter: line 15 column 27 (char 1346)


Fixed file written: perplexity-sample-fixed.json


JSON is valid


In [ ]:
with open("/content/perplexity-sample-fixed.json", "r") as f:
    data = json.load(f)

df = pd.json_normalize(
    data,
    record_path="turns",
    meta=["id", "topic"]
)

print(df.head())
print(df.columns)


        role  \
0       user   
1  assistant   
2       user   
3  assistant   
4       user   

                                                                                                                                                                                                                                     message  \
0                                                                                                                                                                               I'm planning a trip to Japan next spring. Where should I go?   
1                                                                                          Japan in spring is amazing for cherry blossoms! Start with Tokyo for modern vibes, then Kyoto for temples and history. How many days do you have?   
2                                                                                                                                                                                      

In [ ]:
with open("/content/perplexity-sample-fixed.json", "r") as f:
    perplexity_data = json.load(f)

# Flatten conversations
df_perplexity = pd.json_normalize(
    perplexity_data,
    record_path="turns",
    meta=["id", "topic"]
)

df_perplexity.to_csv("/content/perplexity-sample.csv", index=False)

print("perplexity-sample.csv exported")
print(df_perplexity.head())


perplexity-sample.csv exported
        role  \
0       user   
1  assistant   
2       user   
3  assistant   
4       user   

                                                                                                                                                                                                                                     message  \
0                                                                                                                                                                               I'm planning a trip to Japan next spring. Where should I go?   
1                                                                                          Japan in spring is amazing for cherry blossoms! Start with Tokyo for modern vibes, then Kyoto for temples and history. How many days do you have?   
2                                                                                                                                                       

In [ ]:
df = pd.read_csv("perplexity-sample.csv")

# Drop the 'topic' column
df = df.drop(columns=["topic"])

# Rename 'message' column to 'content'
df = df.rename(columns={"message": "content"})

# Save the updated CSV
df.to_csv("perplexity-sample-updated.csv", index=False)

df.head()

,role,content,id
0,user,I'm planning a trip to Japan next spring. Where should I go?,1
1,assistant,"Japan in spring is amazing for cherry blossoms! Start with Tokyo for modern vibes, then Kyoto for temples and history. How many days do you have?",1
2,user,About 10 days. I love food and hiking.,1
3,assistant,"Perfect. Days 1-3: Tokyo (Sushi at Tsukiji, Shibuya Crossing). Day 4: Train to Hakone for hiking Mt. Fuji views and onsen. Days 5-8: Kyoto (Fushimi Inari hike, kaiseki meals). Day 9: Nara for deer park. Day 10: Back to Tokyo. Budget?",1
4,user,"Mid-range, around $2000 excluding flights. Any tips for cherry blossom spots?",1




---


# **Ultrachat Dataset Training**


---



In [ ]:
ds = pd.read_csv("ultrachat_sample.csv")

print(ds.head())
print('\n')
train_ds, test_ds = train_test_split(
    ds,
    test_size=0.2,
    random_state=42
)

print(train_ds.shape)
print(test_ds.shape)
print('\n')
ds.info()


                                                   id       role  \
0  ultrachat_existent_material_release_230420_1141197  assistant   
1  ultrachat_existent_material_release_230420_1141197       user   
2  ultrachat_existent_material_release_230420_1141197  assistant   
3  ultrachat_existent_material_release_230420_1141198       user   
4  ultrachat_existent_material_release_230420_1141198  assistant   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [ ]:
ds.head()


,id,role,content
0,ultrachat_existent_material_release_230420_1141197,assistant,"The treatment protocol for meningitis may involve the administration of antibiotics, such as cefotaxime or ceftriaxone, and possibly steroids to reduce inflammation. Meningitis can result in altered levels of consciousness in patients due to the inflammation affecting the brain and nervous system. Immediate medical attention is necessary to prevent serious complications and potential death."
1,ultrachat_existent_material_release_230420_1141197,user,Can you provide information on the symptoms of tularemia?
2,ultrachat_existent_material_release_230420_1141197,assistant,"Certainly! Tularemia is a bacterial infection caused by Francisella tularensis. Symptoms can vary depending on how the bacteria entered the body, but they can include fever, chills, headache, muscle aches, joint pain, dry cough, difficulty breathing, chest pain, abdominal pain, diarrhea, vomiting, skin ulcers, swollen lymph nodes, and conjunctivitis. If you suspect you have been exposed to tularemia or are experiencing any of these symptoms, seek medical attention immediately."
3,ultrachat_existent_material_release_230420_1141198,user,"Read the passage below and answer the question or follow the instruction: What is the role of TVFreedom.org in advocating for the retransmission system?\n\nPassage: About a day after its 129 broadcast stations went dark and mere hours after FCC chairman Tom Wheeler called for an emergency meeting to reach an end to their dispute, Dish Network and Sinclair Broadcast Group said they have reached an agreement and that restoration of the Sinclair signals to Dish subscribers is underway.\n“We are grateful for the FCC’s work on behalf of consumers to actively broker a productive path forward,” said Jeff Blum, DISH senior VP and deputy general counsel. Terms of the agreement were not disclosed.\nSinclair pulled its signals on Tuesday after a short extension expired and the two parties could not reach an agreement. Dish had claimed it had come to an agreement on pricing for Sinclair’s stations, but that the broadcaster was demanding the satellite giant negotiate carriage for an unnamed cable channel Sinclair hoped to own. Dish also had filed a complaint with the FCC, claiming Sinclair was not negotiating in good faith. As part of its new deal, Dish is asking the FCC to stay action on the complaint as its long-term agreement is being finalized.\nThe chairman also trumpeted the tentative agreement and the FCC&apos;s role in pressing for resolution. ""Today the most extensive retransmission blackout in history ended,"" he Tweeted, followed by: ""The FCC will remain vigilant. Use of the public airwaves is a public trust.""\n""We are pleased to see that five million Dish subscribers are again able to access their local news and programming and most-watched network entertainment and live sporting events with the agreement reached this evening by Sinclair and Dish,"" said Rob Kenny of TVFreedom.org, a broadcaster-backed advocate for the retrans system. ""Local TV newscasts, severe weather updates and emergency alerts and warnings are important to viewers and something they value and rely on daily.""\nThis article was original published on Multichannel.com."
4,ultrachat_existent_material_release_230420_1141198,assistant,What is the role of TVFreedom.org in advocating for the retransmission system?\n\nTVFreedom.org is a broadcaster-backed advocate for the retransmission system.




---


# **Proceeding to ML implementation over the different OpenAI and GenAI based model datasets**

---



In [ ]:
import unicodedata
files = {
    "gemini": "/content/gemini-sample-updated.csv",
    "perplexity": "/content/perplexity-sample-updated.csv",
    "ultrachat": "/content/ultrachat_sample.csv"
}

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[#*_>`~\-]+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

dfs = []

for model_name, path in files.items():
    df = pd.read_csv(path)
    df = df[["id", "role", "content"]]
    df = df[df["role"].isin(["user", "assistant"])]
    df["content_clean"] = df["content"].apply(clean_text)
    df = df[df["content_clean"].str.len() > 10]
    df["model_name"] = model_name
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# SHOW combined dataset
combined_df.head(100)


id       role  \
0   gpt_conv01       user   
1   gpt_conv01  assistant   
2   gpt_conv02       user   
3   gpt_conv02  assistant   
4   gpt_conv03       user   
..         ...        ...   
95  gpt_conv20       user   
96  gpt_conv20  assistant   
97  gpt_conv20       user   
98  gpt_conv20  assistant   
99  gpt_conv20       user   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [ ]:
df = pd.read_csv("combined_dataset.csv")

***Approach one using TFIDF and linguistic features***

In [ ]:
# CONFIGURATION
FILES = {
    "gemini": "gemini-sample-updated.csv",
    "perplexity": "perplexity-sample-updated.csv",
    "ultrachat": "ultrachat_sample.csv"
}


SPEED_TIER = 'fast'


# DATA LOADING & PREPROCESSING
def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKD", text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[#*_>`~\-]+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def load_and_prepare_data(files):

    print("Loading datasets...")
    dfs = []

    for model_name, path in files.items():
        try:
            df = pd.read_csv(path)
            df = df[["id", "role", "content"]]
            df = df[df["role"].isin(["user", "assistant"])]
            df["content_clean"] = df["content"].apply(clean_text)
            df = df[df["content_clean"].str.len() > 10]
            df["model_name"] = model_name
            dfs.append(df)
            print(f"  ✓ Loaded {model_name}: {len(df)} rows")
        except Exception as e:
            print(f"  ✗ Error loading {model_name}: {e}")

    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(combined_df)}")

    return combined_df


# FEATURE EXTRACTION - Using TF-IDF

def compute_tfidf_features(pairs_df):

    print("\nComputing TF-IDF features...")

    # Combine all text for vocabulary
    all_text = pd.concat([
        pairs_df['content_clean_user'],
        pairs_df['content_clean_assistant']
    ]).fillna("")

    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )

    # Fit on all text
    vectorizer.fit(all_text)

    # Transform user and assistant separately
    user_tfidf = vectorizer.transform(pairs_df['content_clean_user'].fillna(""))
    assistant_tfidf = vectorizer.transform(pairs_df['content_clean_assistant'].fillna(""))

    # Compute cosine similarity
    similarities = []
    for i in range(user_tfidf.shape[0]):
        sim = cosine_similarity(user_tfidf[i], assistant_tfidf[i])[0][0]
        similarities.append(sim)

    pairs_df['tfidf_similarity'] = similarities
    print(f"  ✓ TF-IDF similarity computed")

    return pairs_df


# FEATURE EXTRACTION - Using Linguistic Features

def compute_linguistic_features(pairs_df):

    print("\nComputing linguistic features...")

    # Length ratio
    pairs_df['length_ratio'] = (
        pairs_df['content_clean_assistant'].fillna("").str.split().str.len() /
        (pairs_df['content_clean_user'].fillna("").str.split().str.len() + 1)
    )

    # Lexical diversity (unique words / total words)
    pairs_df['assistant_lexical_diversity'] = pairs_df['content_clean_assistant'].apply(
        lambda x: len(set(x.split())) / (len(x.split()) + 1) if isinstance(x, str) and x else 0
    )

    # Vocabulary overlap
    def vocab_overlap(user, assistant):
        if not user or not assistant:
            return 0
        user_words = set(user.split())
        assistant_words = set(assistant.split())
        if not user_words:
            return 0
        return len(user_words & assistant_words) / len(user_words)

    pairs_df['vocab_overlap'] = pairs_df.apply(
        lambda row: vocab_overlap(row['content_clean_user'], row['content_clean_assistant']),
        axis=1
    )

    # Repetition score (duplicate bigrams)
    def repetition_score(text):
        if not text:
            return 0
        words = text.split()
        if len(words) < 2:
            return 0
        bigrams = [f"{words[i]}_{words[i+1]}" for i in range(len(words)-1)]
        if not bigrams:
            return 0
        return 1 - (len(set(bigrams)) / len(bigrams))

    pairs_df['repetition_score'] = pairs_df['content_clean_assistant'].apply(repetition_score)

    # Hedging language detection
    hedging_words = ['maybe', 'perhaps', 'possibly', 'might', 'could', 'probably',
                     'seems', 'appears', 'likely', 'uncertain', 'unclear']

    def hedging_score(text):
        if not text:
            return 0
        words = text.split()
        if not words:
            return 0
        hedge_count = sum(1 for word in words if word in hedging_words)
        return hedge_count / len(words)

    pairs_df['hedging_score'] = pairs_df['content_clean_assistant'].apply(hedging_score)

    # Response specificity (noun/entity density approximation)
    def specificity_score(text):
        if not text:
            return 0
        words = text.split()
        if not words:
            return 0
        # Capitalized words after sentence start as proxy for entities
        capitalized = sum(1 for w in words if len(w) > 2 and w[0].isupper())
        return capitalized / len(words)

    pairs_df['specificity_score'] = pairs_df['content_clean_assistant'].apply(specificity_score)

    print(f"  ✓ Linguistic features computed")

    return pairs_df


# FEATURE EXTRACTION - Using Small Transformer

def compute_transformer_embeddings(pairs_df):

    print("\nComputing transformer embeddings (this may take 15-20 minutes)...")

    try:
        from sentence_transformers import SentenceTransformer

        # Use smallest, fastest model
        model = SentenceTransformer('all-MiniLM-L6-v2')  # Only 80MB, very fast

        # Encode in batches
        print("  Encoding user prompts...")
        user_embs = model.encode(
            pairs_df['content_clean_user'].fillna("").tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        print("  Encoding assistant responses...")
        assistant_embs = model.encode(
            pairs_df['content_clean_assistant'].fillna("").tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        # Compute similarity
        similarities = []
        for i in range(len(user_embs)):
            sim = cosine_similarity([user_embs[i]], [assistant_embs[i]])[0][0]
            similarities.append(sim)

        pairs_df['transformer_similarity'] = similarities

        # Store embeddings for distribution analysis
        pairs_df['assistant_embedding'] = list(assistant_embs)

        print(f"  ✓ Transformer embeddings computed")

    except ImportError:
        print("  ⚠ sentence-transformers not installed, skipping")
        pairs_df['transformer_similarity'] = pairs_df['tfidf_similarity']

    return pairs_df


# DISTRIBUTION-BASED FEATURES:

def compute_distribution_features(pairs_df):

    print("\nComputing distribution features...")

    # For each model, compute centroid and distance
    for model in pairs_df['model_name'].unique():
        model_mask = pairs_df['model_name'] == model

        # Length distribution stats
        model_lengths = pairs_df[model_mask]['length_ratio']
        mean_length = model_lengths.mean()
        std_length = model_lengths.std()

        pairs_df.loc[model_mask, 'length_deviation'] = np.abs(
            pairs_df.loc[model_mask, 'length_ratio'] - mean_length
        ) / (std_length + 1e-6)

        # Similarity distribution
        if 'tfidf_similarity' in pairs_df.columns:
            model_sims = pairs_df[model_mask]['tfidf_similarity']
            mean_sim = model_sims.mean()
            std_sim = model_sims.std()

            pairs_df.loc[model_mask, 'similarity_deviation'] = np.abs(
                pairs_df.loc[model_mask, 'tfidf_similarity'] - mean_sim
            ) / (std_sim + 1e-6)

    print(f"  ✓ Distribution features computed")

    return pairs_df


# HALLUCINATION SCORING

def compute_hallucination_scores(pairs_df, tier='fast'):

    print(f"\nComputing hallucination scores (tier: {tier})...")

    if tier == 'ultra_fast':
        # Only TF-IDF based
        pairs_df['hallucination_score'] = (
            0.7 * (1 - pairs_df['tfidf_similarity']) +
            0.3 * np.clip(pairs_df['length_ratio'] / 5, 0, 1)
        )

    elif tier == 'fast':
        # TF-IDF + linguistic features
        pairs_df['hallucination_score'] = (
            0.30 * (1 - pairs_df['tfidf_similarity']) +
            0.15 * np.clip(pairs_df['length_ratio'] / 5, 0, 1) +
            0.15 * (1 - pairs_df['vocab_overlap']) +
            0.15 * pairs_df['repetition_score'] +
            0.10 * pairs_df['hedging_score'] +
            0.10 * np.clip(pairs_df['similarity_deviation'], 0, 1) +
            0.05 * (1 - pairs_df['assistant_lexical_diversity'])
        )

    elif tier == 'balanced':
        # All features including transformer
        pairs_df['hallucination_score'] = (
            0.25 * (1 - pairs_df['transformer_similarity']) +
            0.20 * (1 - pairs_df['tfidf_similarity']) +
            0.15 * np.clip(pairs_df['length_ratio'] / 5, 0, 1) +
            0.10 * (1 - pairs_df['vocab_overlap']) +
            0.10 * pairs_df['repetition_score'] +
            0.10 * pairs_df['hedging_score'] +
            0.05 * np.clip(pairs_df['similarity_deviation'], 0, 1) +
            0.05 * (1 - pairs_df['assistant_lexical_diversity'])
        )

    # Normalize to 0-1 range
    pairs_df['hallucination_score'] = (
        pairs_df['hallucination_score'] - pairs_df['hallucination_score'].min()
    ) / (pairs_df['hallucination_score'].max() - pairs_df['hallucination_score'].min() + 1e-6)

    print(f"  ✓ Hallucination scores computed")

    return pairs_df




# MAIN PIPELINE
def create_pairs(df):

    print("\nCreating user-assistant pairs...")

    pairs = df.pivot_table(
        index=['model_name', 'id'],
        columns='role',
        values='content_clean',
        aggfunc='first'
    )

    pairs.columns = [f'content_clean_{col}' for col in pairs.columns]
    pairs = pairs.reset_index()

    # Remove pairs with missing data
    pairs = pairs.dropna(subset=['content_clean_user', 'content_clean_assistant'])

    print(f"  ✓ Created {len(pairs)} pairs")

    return pairs

def run_pipeline(files, speed_tier='fast'):
    # Load data
    df = load_and_prepare_data(files)

    # Create pairs
    pairs = create_pairs(df)


    pairs = compute_tfidf_features(pairs)


    if speed_tier in ['fast', 'balanced']:
        pairs = compute_linguistic_features(pairs)
        pairs = compute_distribution_features(pairs)

    if speed_tier == 'balanced':
        pairs = compute_transformer_embeddings(pairs)

    # Compute final scores
    pairs = compute_hallucination_scores(pairs, tier=speed_tier)

    # Generate summary
    print("\n" + "="*70)
    print("RESULTS SUMMARY")
    print("="*70)

    summary = pairs.groupby('model_name')['hallucination_score'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).round(4)

    summary = summary.sort_values('mean')
    print("\nHallucination Scores by Model:")
    print(summary)

    print("\n" + "="*70)

    return pairs, summary



if __name__ == "__main__":
    # Run pipeline
    pairs_df, summary_df = run_pipeline(FILES, speed_tier=SPEED_TIER)

    # Save results
    pairs_df.to_csv('hallucination_scores.csv', index=False)
    summary_df.to_csv('model_comparison.csv')

    print("\n✓ Results saved to:")
    print("  - hallucination_scores.csv")
    print("  - model_comparison.csv")

Loading datasets...
  ✓ Loaded chatgpt: 3132 rows
  ✓ Loaded claude: 1264 rows
  ✓ Loaded gemini: 149 rows
  ✓ Loaded perplexity: 365 rows
  ✓ Loaded ultrachat: 3841 rows

Total rows: 8751

Creating user-assistant pairs...
  ✓ Created 1508 pairs

Computing TF-IDF features...
  ✓ TF-IDF similarity computed

Computing linguistic features...
  ✓ Linguistic features computed

Computing distribution features...
  ✓ Distribution features computed

Computing hallucination scores (tier: fast)...
  ✓ Hallucination scores computed

RESULTS SUMMARY

Hallucination Scores by Model:
            count    mean     std     min     max
model_name                                       
ultrachat     812  0.3183  0.1832  0.0089  0.8662
chatgpt       395  0.4993  0.1895  0.0070  1.0000
gemini         20  0.5160  0.1164  0.3438  0.7475
perplexity     49  0.5488  0.0320  0.4106  0.6574
claude        232  0.5491  0.1877  0.0000  0.9532


✓ Results saved to:
  - hallucination_scores.csv
  - model_comparison.cs

***Approach two using  MPNet adn DeBERTa***

In [ ]:
# CONFIGURATION

FILES = {
    "gemini": "gemini-sample-updated.csv",
    "perplexity": "perplexity-sample-updated.csv",
    "ultrachat": "ultrachat_sample.csv"
}


USE_NLI_MODEL = True
USE_SEMANTIC_MODEL = True


# DATA LOADING & PREPROCESSING

def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKD", text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[#*_>`~\-]+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def load_and_prepare_data(files):
    print("\n")
    print("LOADING DATASETS")

    dfs = []

    for model_name, path in files.items():
        try:
            df = pd.read_csv(path)
            df = df[["id", "role", "content"]]
            df = df[df["role"].isin(["user", "assistant"])]
            df["content_clean"] = df["content"].apply(clean_text)
            df = df[df["content_clean"].str.len() > 10]
            df["model_name"] = model_name
            dfs.append(df)
            print(f"✓ {model_name:15} {len(df):6,} rows")
        except Exception as e:
            print(f"✗ {model_name:15} Error: {e}")

    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"\n{'Total:':<15} {len(combined_df):6,} rows")


    return combined_df


# SEMANTIC SIMILARITY - MPNet

def compute_semantic_similarity(pairs_df):


    print("COMPUTING SEMANTIC SIMILARITY (MPNet)")


    try:
        from sentence_transformers import SentenceTransformer

        # Use paraphrase-mpnet-base-v2
        model = SentenceTransformer('paraphrase-mpnet-base-v2')
        print("✓ Model loaded: paraphrase-mpnet-base-v2")

        # Encode user prompts
        print("→ Encoding user prompts...")
        user_embs = model.encode(
            pairs_df['content_clean_user'].fillna("").tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        # Encode assistant responses
        print("→ Encoding assistant responses...")
        assistant_embs = model.encode(
            pairs_df['content_clean_assistant'].fillna("").tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        # Compute cosine similarity
        similarities = np.array([
            cosine_similarity([user_embs[i]], [assistant_embs[i]])[0][0]
            for i in range(len(user_embs))
        ])

        pairs_df['semantic_similarity'] = similarities
        pairs_df['assistant_embedding'] = list(assistant_embs)

        print(f"✓ Semantic similarity computed")
        print(f"  Mean: {similarities.mean():.4f}, Std: {similarities.std():.4f}")

    except ImportError:
        print("⚠ sentence-transformers not installed. Run: pip install sentence-transformers")
        pairs_df['semantic_similarity'] = 0.5

    return pairs_df


# NLI-BASED HALLUCINATION DETECTION - DeBERTa (SOTA)

def compute_nli_scores(pairs_df):


    print("COMPUTING NLI SCORES (DeBERTa-v3)")


    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch

        model_name = "microsoft/deberta-v3-large"
        print(f"✓ Loading {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=3  # entailment, neutral, contradiction
        )


        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.eval()
        print(f"✓ Model loaded on {device}")

        entailment_scores = []
        contradiction_scores = []
        neutral_scores = []

        batch_size = 16
        total = len(pairs_df)

        print(f"→ Processing {total} pairs...")

        with torch.no_grad():
            for i in range(0, total, batch_size):
                batch = pairs_df.iloc[i:i+batch_size]

                # Create premise-hypothesis pairs for NLI
                premises = batch['content_clean_user'].fillna("").tolist()
                hypotheses = batch['content_clean_assistant'].fillna("").tolist()

                # Tokenize
                inputs = tokenizer(
                    premises,
                    hypotheses,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt"
                ).to(device)

                # Get predictions
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

                # Extract scores (typically: 0=entailment, 1=neutral, 2=contradiction)
                entailment_scores.extend(probs[:, 0])
                neutral_scores.extend(probs[:, 1])
                contradiction_scores.extend(probs[:, 2])

                if (i // batch_size + 1) % 10 == 0:
                    print(f"  Processed {min(i+batch_size, total)}/{total} pairs")

        pairs_df['nli_entailment'] = entailment_scores
        pairs_df['nli_neutral'] = neutral_scores
        pairs_df['nli_contradiction'] = contradiction_scores

        # Faithfulness score: High entailment = faithful, High contradiction = hallucination
        pairs_df['nli_faithfulness'] = pairs_df['nli_entailment'] - pairs_df['nli_contradiction']

        print("✓ NLI scores computed")
        print(f"  Mean entailment: {np.mean(entailment_scores):.4f}")
        print(f"  Mean contradiction: {np.mean(contradiction_scores):.4f}")

    except Exception as e:
        print(f"⚠ NLI computation failed: {e}")
        print("  Continuing without NLI features...")
        pairs_df['nli_entailment'] = 0.5
        pairs_df['nli_contradiction'] = 0.5
        pairs_df['nli_neutral'] = 0.0
        pairs_df['nli_faithfulness'] = 0.0

    return pairs_df


# LINGUISTIC FEATURES

def compute_linguistic_features(pairs_df):


    print("COMPUTING LINGUISTIC FEATURES")


    # 1. Length ratio
    pairs_df['user_length'] = pairs_df['content_clean_user'].fillna("").str.split().str.len()
    pairs_df['assistant_length'] = pairs_df['content_clean_assistant'].fillna("").str.split().str.len()
    pairs_df['length_ratio'] = pairs_df['assistant_length'] / (pairs_df['user_length'] + 1)

    # 2. Lexical diversity
    pairs_df['lexical_diversity'] = pairs_df['content_clean_assistant'].apply(
        lambda x: len(set(x.split())) / (len(x.split()) + 1) if x else 0
    )

    # 3. Vocabulary overlap (key for detecting grounding)
    def vocab_overlap(user, assistant):
        if not user or not assistant:
            return 0
        user_words = set(user.split())
        assistant_words = set(assistant.split())
        if not user_words:
            return 0
        return len(user_words & assistant_words) / len(user_words)

    pairs_df['vocab_overlap'] = pairs_df.apply(
        lambda row: vocab_overlap(row['content_clean_user'], row['content_clean_assistant']),
        axis=1
    )

    # 4. Repetition score (detecting degeneration)
    def repetition_score(text):
        if not text:
            return 0
        words = text.split()
        if len(words) < 2:
            return 0
        bigrams = [f"{words[i]}_{words[i+1]}" for i in range(len(words)-1)]
        if not bigrams:
            return 0
        return 1 - (len(set(bigrams)) / len(bigrams))

    pairs_df['repetition_score'] = pairs_df['content_clean_assistant'].apply(repetition_score)

    # 5. Hedging language (uncertainty markers)
    hedging_words = {'maybe', 'perhaps', 'possibly', 'might', 'could', 'probably',
                     'seems', 'appears', 'likely', 'uncertain', 'unclear', 'think',
                     'believe', 'guess', 'assume', 'suppose'}

    def hedging_score(text):
        if not text:
            return 0
        words = text.split()
        if not words:
            return 0
        hedge_count = sum(1 for word in words if word in hedging_words)
        return hedge_count / len(words)

    pairs_df['hedging_score'] = pairs_df['content_clean_assistant'].apply(hedging_score)

    # 6. Specificity (entity/proper noun density proxy)
    def specificity_score(text):
        if not text:
            return 0
        words = text.split()
        if not words:
            return 0
        # Longer words tend to be more specific
        long_words = sum(1 for w in words if len(w) >= 7)
        return long_words / len(words)

    pairs_df['specificity_score'] = pairs_df['content_clean_assistant'].apply(specificity_score)

    print("✓ Linguistic features computed")
    print(f"  Mean length ratio: {pairs_df['length_ratio'].mean():.2f}")
    print(f"  Mean vocab overlap: {pairs_df['vocab_overlap'].mean():.3f}")
    print(f"  Mean hedging: {pairs_df['hedging_score'].mean():.4f}")

    return pairs_df


# DISTRIBUTION-BASED ANOMALY DETECTION
def compute_distribution_features(pairs_df):


    print("COMPUTING DISTRIBUTION FEATURES")


    for model in pairs_df['model_name'].unique():
        model_mask = pairs_df['model_name'] == model

        # Length deviation from model norm
        model_lengths = pairs_df[model_mask]['length_ratio']
        mean_length = model_lengths.mean()
        std_length = model_lengths.std()

        pairs_df.loc[model_mask, 'length_z_score'] = np.abs(
            (pairs_df.loc[model_mask, 'length_ratio'] - mean_length) / (std_length + 1e-6)
        )

        # Semantic similarity deviation
        if 'semantic_similarity' in pairs_df.columns:
            model_sims = pairs_df[model_mask]['semantic_similarity']
            mean_sim = model_sims.mean()
            std_sim = model_sims.std()

            pairs_df.loc[model_mask, 'similarity_z_score'] = np.abs(
                (pairs_df.loc[model_mask, 'semantic_similarity'] - mean_sim) / (std_sim + 1e-6)
            )

    # Normalize z-scores to 0-1 range
    pairs_df['length_anomaly'] = np.clip(pairs_df['length_z_score'] / 3, 0, 1)
    pairs_df['similarity_anomaly'] = np.clip(pairs_df['similarity_z_score'] / 3, 0, 1)

    print("✓ Distribution features computed")

    return pairs_df


# COMPREHENSIVE HALLUCINATION SCORING
def compute_hallucination_scores(pairs_df):


    print("COMPUTING FINAL HALLUCINATION SCORES")


    # Component scores (0-1 range, higher = more hallucination)
    components = {}

    # 1. NLI-based (most reliable for hallucination detection)
    if 'nli_contradiction' in pairs_df.columns:
        components['nli'] = pairs_df['nli_contradiction'] * 0.35
    else:
        components['nli'] = 0

    # 2. Semantic dissimilarity
    if 'semantic_similarity' in pairs_df.columns:
        components['semantic'] = (1 - pairs_df['semantic_similarity']) * 0.25
    else:
        components['semantic'] = 0

    # 3. Lack of grounding (low vocab overlap)
    components['grounding'] = (1 - pairs_df['vocab_overlap']) * 0.15

    # 4. Hedging/uncertainty
    components['uncertainty'] = pairs_df['hedging_score'] * 10 * 0.10  # Scale up

    # 5. Repetition (degeneration)
    components['repetition'] = pairs_df['repetition_score'] * 0.05

    # 6. Semantic anomaly
    if 'similarity_anomaly' in pairs_df.columns:
        components['anomaly'] = pairs_df['similarity_anomaly'] * 0.05
    else:
        components['anomaly'] = 0

    # 7. Length anomaly
    if 'length_anomaly' in pairs_df.columns:
        components['length'] = pairs_df['length_anomaly'] * 0.05
    else:
        components['length'] = 0

    # Combine all components
    pairs_df['hallucination_score'] = sum(components.values())

    # Clip to ensure 0-1 range
    pairs_df['hallucination_score'] = np.clip(pairs_df['hallucination_score'], 0, 1)

    # Store component contributions for analysis
    for name, values in components.items():
        pairs_df[f'component_{name}'] = values

    print("✓ Hallucination scores computed")
    print("\nScore distribution:")
    print(pairs_df['hallucination_score'].describe().round(4))

    return pairs_df


# DEVIATION ANALYSIS
def compute_deviation_metrics(pairs_df):


    print("COMPUTING DEVIATION METRICS")


    # 1. Semantic deviation (1 - similarity as percentage)
    if 'semantic_similarity' in pairs_df.columns:
        pairs_df['semantic_deviation_pct'] = (1 - pairs_df['semantic_similarity']) * 100
    else:
        pairs_df['semantic_deviation_pct'] = 0

    # 2. Vocabulary deviation (ungrounded content percentage)
    pairs_df['vocab_deviation_pct'] = (1 - pairs_df['vocab_overlap']) * 100

    # 3. Overall deviation (composite)
    pairs_df['overall_deviation_pct'] = pairs_df['hallucination_score'] * 100

    print("✓ Deviation metrics computed")

    return pairs_df


# MAIN PIPELINE
def create_pairs(df):


    print("CREATING USER-ASSISTANT PAIRS")


    pairs = df.pivot_table(
        index=['model_name', 'id'],
        columns='role',
        values='content_clean',
        aggfunc='first'
    )

    pairs.columns = [f'content_clean_{col}' for col in pairs.columns]
    pairs = pairs.reset_index()

    # Remove pairs with missing data
    pairs = pairs.dropna(subset=['content_clean_user', 'content_clean_assistant'])

    print(f"✓ Created {len(pairs):,} valid pairs")

    # Distribution by model
    print("\nPairs per model:")
    for model, count in pairs['model_name'].value_counts().sort_index().items():
        print(f"  {model:15} {count:6,} pairs")

    return pairs

def run_pipeline(files):
    print("\nModels used:")
    print("  • MPNet: Semantic similarity")
    print("  • DeBERTa-v3: NLI entailment/contradiction (SOTA)")

    # Load data
    df = load_and_prepare_data(files)

    # Create pairs
    pairs = create_pairs(df)

    # Core features
    pairs = compute_linguistic_features(pairs)

    # Advanced features
    if USE_SEMANTIC_MODEL:
        pairs = compute_semantic_similarity(pairs)

    if USE_NLI_MODEL:
        pairs = compute_nli_scores(pairs)

    # Distribution analysis
    pairs = compute_distribution_features(pairs)

    # Final scores
    pairs = compute_hallucination_scores(pairs)
    pairs = compute_deviation_metrics(pairs)

    # Generate comprehensive summary
    generate_summary(pairs)

    return pairs

def generate_summary(pairs_df):

    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)

    # 1. Overall statistics
    print("\n1. HALLUCINATION SCORES BY MODEL")


    summary = pairs_df.groupby('model_name').agg({
        'hallucination_score': ['count', 'mean', 'std', 'min', 'max'],
        'overall_deviation_pct': 'mean',
        'semantic_deviation_pct': 'mean',
        'vocab_deviation_pct': 'mean'
    }).round(4)

    summary.columns = ['_'.join(col).strip() for col in summary.columns]
    summary = summary.sort_values('hallucination_score_mean')

    print(summary)

    # 2. Deviation percentages
    print("\n2. DEVIATION FROM USER PROMPTS (Percentage)")


    deviation_summary = pairs_df.groupby('model_name').agg({
        'overall_deviation_pct': ['mean', 'std'],
        'semantic_deviation_pct': ['mean', 'std'],
        'vocab_deviation_pct': ['mean', 'std']
    }).round(2)

    deviation_summary.columns = ['_'.join(col).strip() for col in deviation_summary.columns]
    deviation_summary = deviation_summary.sort_values('overall_deviation_pct_mean')

    print(deviation_summary)

    # 3. Component analysis
    print("\n3. HALLUCINATION COMPONENTS BREAKDOWN")


    component_cols = [col for col in pairs_df.columns if col.startswith('component_')]
    if component_cols:
        comp_summary = pairs_df.groupby('model_name')[component_cols].mean().round(4)
        comp_summary.columns = [col.replace('component_', '') for col in comp_summary.columns]
        print(comp_summary)

    # 4. Rankings
    print("\n4. MODEL RANKINGS (Best to Worst)")


    rankings = pairs_df.groupby('model_name')['hallucination_score'].mean().sort_values()

    for rank, (model, score) in enumerate(rankings.items(), 1):
        score_pct = score * 100
        quality = "Excellent" if score < 0.3 else "Good" if score < 0.5 else "Fair" if score < 0.7 else "Poor"
        print(f"  {rank}. {model:15} Score: {score:.4f} ({score_pct:5.2f}%) - {quality}")

    print("\n" + "="*80)

    # Statistical significance
    print("\n5. STATISTICAL ANALYSIS")


    from scipy.stats import f_oneway
    groups = [pairs_df[pairs_df['model_name'] == m]['hallucination_score'].values
              for m in pairs_df['model_name'].unique()]
    f_stat, p_val = f_oneway(*groups)

    print(f"ANOVA F-statistic: {f_stat:.4f}")
    #print(f"P-value: {p_val:.6f}")
    #print(f"Significant difference between models: {'YES' if p_val < 0.05 else 'NO'}")




# EXECUTE
if __name__ == "__main__":

    pairs_df = run_pipeline(FILES)


    print("\nSaving results...")


    pairs_df.to_csv('hallucination_analysis.csv', index=False)
    print("✓ hallucination_analysis.csv saved")

    # Model comparison
    model_summary = pairs_df.groupby('model_name').agg({
        'hallucination_score': ['mean', 'std', 'min', 'max'],
        'overall_deviation_pct': 'mean',
        'semantic_deviation_pct': 'mean',
        'vocab_deviation_pct': 'mean',
        'nli_contradiction': 'mean',
        'semantic_similarity': 'mean'
    }).round(4)

    model_summary.to_csv('model_comparison.csv')
    print("✓ model_comparison.csv saved")


    print("PIPELINE COMPLETE!")



Models used:
  • MPNet: Semantic similarity
  • DeBERTa-v3: NLI entailment/contradiction (SOTA)


LOADING DATASETS
✓ chatgpt          3,132 rows
✓ claude           1,264 rows
✓ gemini             149 rows
✓ perplexity         365 rows
✓ ultrachat        3,841 rows

Total:           8,751 rows
CREATING USER-ASSISTANT PAIRS
✓ Created 1,508 valid pairs

Pairs per model:
  chatgpt            395 pairs
  claude             232 pairs
  gemini              20 pairs
  perplexity          49 pairs
  ultrachat          812 pairs
COMPUTING LINGUISTIC FEATURES
✓ Linguistic features computed
  Mean length ratio: 12.05
  Mean vocab overlap: 0.437
  Mean hedging: 0.0020
COMPUTING SEMANTIC SIMILARITY (MPNet)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded on cuda
→ Processing 1508 pairs...
  Processed 160/1508 pairs
  Processed 320/1508 pairs
  Processed 480/1508 pairs
  Processed 640/1508 pairs
  Processed 800/1508 pairs
  Processed 960/1508 pairs
  Processed 1120/1508 pairs
  Processed 1280/1508 pairs
  Processed 1440/1508 pairs
✓ NLI scores computed
  Mean entailment: 0.3554
  Mean contradiction: 0.3760
COMPUTING DISTRIBUTION FEATURES
✓ Distribution features computed
COMPUTING FINAL HALLUCINATION SCORES
✓ Hallucination scores computed

Score distribution:
count    1508.0000
mean        0.3389
std         0.0703
min         0.1969
25%         0.2866
50%         0.3314
75%         0.3826
max         0.5911
Name: hallucination_score, dtype: float64
COMPUTING DEVIATION METRICS
✓ Deviation metrics computed

RESULTS SUMMARY

1. HALLUCINATION SCORES BY MODEL
            hallucination_score_count  hallucination_score_mean  \
model_name                                                        
chatgpt                           39

***Approach 3 using semantics with SM25 and MMR***

In [ ]:

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

CONFIG = {
    'temperature': 0.2,
    'max_tokens': 512,
    'top_k': 5,
    'similarity_threshold': 0.55,
    'mmr_lambda': 0.7
}

class HallucinationMetrics:

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model

    def calculate_faithfulness(self, response: str, context: str) -> float:
        claims = self._extract_claims(response)
        if not claims:
            return 0.0

        supported_claims = 0
        for claim in claims:
            if self._is_claim_supported(claim, context):
                supported_claims += 1

        return supported_claims / len(claims)

    def calculate_context_relevance(self, query: str, contexts) -> float:
        if not contexts:
            return 0.0

        if isinstance(contexts, str):
            contexts = [contexts]
        elif not isinstance(contexts, list):
            contexts = list(contexts)

        contexts = [c for c in contexts if c and isinstance(c, str) and c.strip()]

        if not contexts:
            return 0.0

        query_embedding = self.embedding_model.encode(query, convert_to_tensor=True)
        context_embeddings = self.embedding_model.encode(contexts, convert_to_tensor=True)

        similarities = util.cos_sim(query_embedding, context_embeddings)[0]
        return float(similarities.mean())

    def calculate_citation_accuracy(self, response: str, available_contexts) -> float:
        if isinstance(available_contexts, str):
            available_contexts = [available_contexts]
        elif not isinstance(available_contexts, list):
            available_contexts = list(available_contexts)

        citations = self._extract_citations(response)
        if not citations:
            return 1.0

        verified_citations = 0
        for citation in citations:
            if self._verify_citation(citation, available_contexts):
                verified_citations += 1

        return verified_citations / len(citations)

    def calculate_answer_relevance(self, query: str, response: str) -> float:
        query_embedding = self.embedding_model.encode(query, convert_to_tensor=True)
        response_embedding = self.embedding_model.encode(response, convert_to_tensor=True)

        similarity = util.cos_sim(query_embedding, response_embedding)[0][0]
        return float(similarity)

    def calculate_hallucination_index(self, faithfulness: float, context_rel: float,
                                     citation_acc: float, answer_rel: float) -> float:
        weights = {'faithfulness': 0.35, 'context': 0.25, 'citation': 0.25, 'answer': 0.15}

        score = (
            (1 - faithfulness) * weights['faithfulness'] +
            (1 - context_rel) * weights['context'] +
            (1 - citation_acc) * weights['citation'] +
            (1 - answer_rel) * weights['answer']
        )

        return score * 100

    def _extract_claims(self, text: str) -> List[str]:
        sentences = sent_tokenize(text)
        claims = [s for s in sentences if len(s.split()) > 5 and not s.strip().endswith('?')]
        return claims

    def _is_claim_supported(self, claim: str, context: str) -> bool:
        claim_embedding = self.embedding_model.encode(claim, convert_to_tensor=True)
        context_sentences = sent_tokenize(context)

        if not context_sentences:
            return False

        context_embeddings = self.embedding_model.encode(context_sentences, convert_to_tensor=True)
        similarities = util.cos_sim(claim_embedding, context_embeddings)[0]

        return float(similarities.max()) > CONFIG['similarity_threshold']

    def _extract_citations(self, text: str) -> List[str]:
        patterns = [
            r'\[\d+\]',
            r'\(\d+\)',
            r'Section\s+\d+',
            r'Article\s+\d+',
            r'Act\s+\d+'
        ]

        citations = []
        for pattern in patterns:
            citations.extend(re.findall(pattern, text, re.IGNORECASE))

        return list(set(citations))

    def _verify_citation(self, citation: str, contexts: List[str]) -> bool:
        citation_clean = citation.lower().strip('[]() ')

        for context in contexts:
            if citation_clean in context.lower():
                return True

        return False


class RetrievalSystem:

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.documents = []
        self.document_embeddings = None
        self.bm25 = None

    def index_documents(self, documents: List[str]):
        self.documents = documents

        print("Creating document embeddings...")
        self.document_embeddings = self.embedding_model.encode(
            documents,
            convert_to_tensor=True,
            show_progress_bar=True
        )

        print("Creating BM25 index...")
        tokenized_docs = [word_tokenize(doc.lower()) for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)

    def retrieve_semantic(self, query: str, top_k: int = 5) -> List[Tuple[str, float]]:
        query_embedding = self.embedding_model.encode(query, convert_to_tensor=True)
        similarities = util.cos_sim(query_embedding, self.document_embeddings)[0]

        top_results = similarities.topk(k=min(top_k, len(self.documents)))

        results = []
        for score, idx in zip(top_results.values, top_results.indices):
            if float(score) >= CONFIG['similarity_threshold']:
                results.append((self.documents[int(idx)], float(score)))

        return results

    def retrieve_bm25(self, query: str, top_k: int = 5) -> List[Tuple[str, float]]:
        tokenized_query = word_tokenize(query.lower())
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = np.argsort(scores)[-top_k:][::-1]

        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append((self.documents[idx], float(scores[idx])))

        return results

    def retrieve_hybrid(self, query: str, top_k: int = 5) -> List[str]:
        semantic_results = self.retrieve_semantic(query, top_k * 2)
        bm25_results = self.retrieve_bm25(query, top_k * 2)

        rrf_scores = defaultdict(float)
        k = 60

        for rank, (doc, _) in enumerate(semantic_results, 1):
            rrf_scores[doc] += 1 / (k + rank)

        for rank, (doc, _) in enumerate(bm25_results, 1):
            rrf_scores[doc] += 1 / (k + rank)

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in sorted_docs[:top_k]]

    def retrieve_mmr(self, query: str, top_k: int = 5, lambda_param: float = 0.7) -> List[str]:
        query_embedding = self.embedding_model.encode(query, convert_to_tensor=True)

        candidates = self.retrieve_semantic(query, top_k * 3)
        if not candidates:
            return []

        selected = []
        candidate_docs = [doc for doc, _ in candidates]
        candidate_embeddings = self.embedding_model.encode(candidate_docs, convert_to_tensor=True)

        selected.append(candidate_docs[0])
        remaining_indices = list(range(1, len(candidate_docs)))

        while len(selected) < top_k and remaining_indices:
            mmr_scores = []

            for idx in remaining_indices:
                relevance = util.cos_sim(query_embedding, candidate_embeddings[idx])[0][0]

                if selected:
                    selected_embeddings = self.embedding_model.encode(selected, convert_to_tensor=True)
                    redundancy = util.cos_sim(candidate_embeddings[idx], selected_embeddings).max()
                else:
                    redundancy = 0

                mmr = lambda_param * relevance - (1 - lambda_param) * redundancy
                mmr_scores.append((idx, float(mmr)))

            best_idx, _ = max(mmr_scores, key=lambda x: x[1])
            selected.append(candidate_docs[best_idx])
            remaining_indices.remove(best_idx)

        return selected


class HallucinationEvaluator:

    def __init__(self, embedding_model):
        self.metrics = HallucinationMetrics(embedding_model)
        self.retrieval = RetrievalSystem(embedding_model)

    def evaluate_dataset(self, df: pd.DataFrame, use_retrieval: bool = True) -> pd.DataFrame:
        results = []

        if use_retrieval and 'content_clean' in df.columns:
            print("Indexing documents for retrieval...")
            self.retrieval.index_documents(df['content_clean'].dropna().tolist())

        print(f"\nEvaluating {len(df)} samples...")

        for idx, row in tqdm(df.iterrows(), total=len(df)):
            try:
                content = str(row['content'])
                content_clean = str(row.get('content_clean', ''))

                query = sent_tokenize(content)[0] if sent_tokenize(content) else content[:200]

                if use_retrieval and content_clean:
                    retrieved_contexts = self.retrieval.retrieve_mmr(query, top_k=CONFIG['top_k'])
                    context = ' '.join(retrieved_contexts) if retrieved_contexts else content_clean
                    contexts_list = retrieved_contexts if retrieved_contexts else [content_clean]
                else:
                    context = content_clean if content_clean else content
                    contexts_list = [context]

                faithfulness = self.metrics.calculate_faithfulness(content, context)
                context_relevance = self.metrics.calculate_context_relevance(query, contexts_list)
                citation_accuracy = self.metrics.calculate_citation_accuracy(content, contexts_list)
                answer_relevance = self.metrics.calculate_answer_relevance(query, content)
                hallucination_index = self.metrics.calculate_hallucination_index(
                    faithfulness, context_relevance, citation_accuracy, answer_relevance
                )

                results.append({
                    'id': row['id'],
                    'model_name': row['model_name'],
                    'faithfulness': faithfulness,
                    'context_relevance': context_relevance,
                    'citation_accuracy': citation_accuracy,
                    'answer_relevance': answer_relevance,
                    'hallucination_index': hallucination_index
                })

            except Exception as e:
                print(f"Error processing row {idx}: {str(e)}")
                results.append({
                    'id': row['id'],
                    'model_name': row['model_name'],
                    'faithfulness': 0.0,
                    'context_relevance': 0.0,
                    'citation_accuracy': 0.0,
                    'answer_relevance': 0.0,
                    'hallucination_index': 100.0
                })

        return pd.DataFrame(results)


def analyze_results(results_df: pd.DataFrame):
    print("HALLUCINATION EVALUATION RESULTS")

    print("\n OVERALL STATISTICS")
    metrics = ['faithfulness', 'context_relevance', 'citation_accuracy',
               'answer_relevance', 'hallucination_index']

    for metric in metrics:
        mean_val = results_df[metric].mean()
        std_val = results_df[metric].std()
        print(f"{metric.replace('_', ' ').title():25s}: {mean_val:.4f} ± {std_val:.4f}")

    print("\n PER-MODEL STATISTICS")

    for model in results_df['model_name'].unique():
        model_data = results_df[results_df['model_name'] == model]
        print(f"\n {model.upper()}")

        for metric in metrics:
            mean_val = model_data[metric].mean()
            print(f"  {metric.replace('_', ' ').title():23s}: {mean_val:.4f}")

    print("\n MODEL RANKINGS (by Hallucination Index - lower is better)")

    model_scores = results_df.groupby('model_name')['hallucination_index'].mean().sort_values()
    for rank, (model, score) in enumerate(model_scores.items(), 1):
        print(f"{rank}. {model:15s}: {score:.4f}")

    print("\n QUALITY DISTRIBUTION")

    def categorize_quality(hi):
        if hi < 25:
            return "Excellent"
        elif hi < 50:
            return "Good"
        elif hi < 75:
            return "Fair"
        else:
            return "Poor"

    results_df['quality_category'] = results_df['hallucination_index'].apply(categorize_quality)
    quality_dist = results_df['quality_category'].value_counts()

    for category in ['Excellent', 'Good', 'Fair', 'Poor']:
        if category in quality_dist:
            count = quality_dist[category]
            pct = (count / len(results_df)) * 100
            print(f"{category:10s}: {count:4d} ({pct:5.1f}%)")

    return model_scores


def save_results(results_df: pd.DataFrame, filename: str = 'hallucination_evaluation_results.csv'):
    results_df.to_csv(filename, index=False)
    print(f"\n Results saved to: {filename}")


def main():
    print(" Starting Hallucination Detection Evaluation Pipeline")

    print("\n Loading dataset...")
    df = pd.read_csv('combined_dataset.csv')

    print(f"✓ Loaded {len(df)} samples")
    print(f"✓ Models: {df['model_name'].unique().tolist()}")
    print(f"✓ Columns: {df.columns.tolist()}")

    print("\n Initializing evaluation pipeline...")
    evaluator = HallucinationEvaluator(embedding_model)

    print("\n Running evaluation...")
    results_df = evaluator.evaluate_dataset(df, use_retrieval=True)

    model_rankings = analyze_results(results_df)

    save_results(results_df)

    print("\n Evaluation complete!")

    return results_df, model_rankings


if __name__ == "__main__":
    results_df, model_rankings = main()

 Starting Hallucination Detection Evaluation Pipeline

 Loading dataset...
✓ Loaded 8742 samples
✓ Models: ['chatgpt', 'claude', 'gemini', 'perplexity', 'ultrachat']
✓ Columns: ['id', 'role', 'content', 'content_clean', 'model_name']

 Initializing evaluation pipeline...

 Running evaluation...
Indexing documents for retrieval...
Creating document embeddings...


Batches:   0%|          | 0/274 [00:00<?, ?it/s]

Creating BM25 index...

Evaluating 8742 samples...


  0%|          | 0/8742 [00:00<?, ?it/s]

HALLUCINATION EVALUATION RESULTS

 OVERALL STATISTICS
Faithfulness             : 0.5145 ± 0.4103
Context Relevance        : 0.7131 ± 0.1922
Citation Accuracy        : 0.9565 ± 0.2039
Answer Relevance         : 0.8305 ± 0.2406
Hallucination Index      : 27.7940 ± 17.6030

 PER-MODEL STATISTICS

 CHATGPT
  Faithfulness           : 0.6184
  Context Relevance      : 0.6877
  Citation Accuracy      : 0.9272
  Answer Relevance       : 0.7931
  Hallucination Index    : 26.0855

 CLAUDE
  Faithfulness           : 0.5734
  Context Relevance      : 0.7098
  Citation Accuracy      : 0.9045
  Answer Relevance       : 0.7940
  Hallucination Index    : 27.6665

 GEMINI
  Faithfulness           : 0.5794
  Context Relevance      : 0.6080
  Citation Accuracy      : 0.9933
  Answer Relevance       : 0.6533
  Hallucination Index    : 29.8872

 PERPLEXITY
  Faithfulness           : 0.1434
  Context Relevance      : 0.8102
  Citation Accuracy      : 1.0000
  Answer Relevance       : 0.9171
  Hallucination 

***Adding Evaluation metrics and Hallucination Detection Verification***

In [ ]:
import torch
if torch.cuda.is_available():
    device = 0
    print(f"✓ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = -1
    print("⚠ No GPU detected, using CPU")


# STATE-OF-THE-ART HALLUCINATION DETECTOR

class StateOfTheArtHallucinationDetector:

    def __init__(self, use_gpu: bool = True):

        # Determine device
        if use_gpu and torch.cuda.is_available():
            self.device = 0
            print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
        else:
            self.device = -1
            print("✓ Using CPU")

        # 1. Semantic Similarity Model (Bi-encoder)
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-mpnet-base-v2')
        if self.device == 0:
            self.embedding_model = self.embedding_model.to('cuda')

        # 2. NLI Model for logical verification
        print("Loading NLI model...")
        self.nli_model = pipeline(
            "text-classification",
            model="facebook/bart-large-mnli",
            device=self.device,
            batch_size=16 if self.device == 0 else 8  # Larger batch on GPU
        )

        # 3. Cross-Encoder for precise relevance
        print("Loading cross-encoder...")
        self.cross_encoder = CrossEncoder(
            'cross-encoder/ms-marco-MiniLM-L-6-v2',
            device='cuda' if self.device == 0 else 'cpu'
        )

        # 4. Knowledge Graph Verifier
        self.kg_verifier = MockKnowledgeGraphVerifier()

        print("✓ All models loaded successfully!\n")

        # Performance optimization
        if self.device == 0:
            torch.cuda.empty_cache()
            print(f"✓ GPU Memory allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
            print(f"✓ GPU Memory cached: {torch.cuda.memory_reserved(0)/1e9:.2f} GB\n")

    def detect_hallucination(self, claim: str, context: str) -> Dict:

        try:
            # Stage 1: Semantic Similarity (fast filter)
            sem_score = self.semantic_similarity(claim, context)

            # Stage 2: NLI Verification (logical check)
            nli_score = self.nli_verify(claim, context)

            # Stage 3: Cross-Encoder Precision (detailed relevance)
            cross_score = self.cross_encoder_score(claim, context)

            # Stage 4: Knowledge Graph Check (for factual claims)
            kg_score = self.kg_verify(claim) if self.is_factual_claim(claim) else 1.0

            # Weighted Ensemble
            final_score = (
                0.25 * sem_score +
                0.35 * nli_score +
                0.25 * cross_score +
                0.15 * kg_score
            )

            return {
                'final_score': final_score,
                'is_hallucination': final_score < 0.5,  # Threshold
                'confidence': abs(final_score - 0.5) * 2,  # How confident (0-1)
                'breakdown': {
                    'semantic': sem_score,
                    'nli': nli_score,
                    'cross_encoder': cross_score,
                    'knowledge_graph': kg_score
                }
            }

        except Exception as e:
            # Catch-all error handler
            print(f"\n⚠ Detection error: {str(e)[:100]}")
            return {
                'final_score': 0.5,
                'is_hallucination': False,
                'confidence': 0.0,
                'breakdown': {
                    'semantic': 0.5,
                    'nli': 0.5,
                    'cross_encoder': 0.5,
                    'knowledge_graph': 0.5
                }
            }

    def semantic_similarity(self, claim: str, context: str) -> float:
        try:
            # Truncate if too long (safety measure)
            max_length = 5000
            claim = claim[:max_length] if len(claim) > max_length else claim
            context = context[:max_length] if len(context) > max_length else context

            claim_emb = self.embedding_model.encode(claim, convert_to_tensor=True)

            # Split context into sentences for better matching
            context_sentences = sent_tokenize(context)
            if not context_sentences:
                return 0.0

            # Process in smaller batches to avoid memory issues
            batch_size = 32
            all_similarities = []

            for i in range(0, len(context_sentences), batch_size):
                batch = context_sentences[i:i+batch_size]
                context_embs = self.embedding_model.encode(batch, convert_to_tensor=True)
                similarities = util.cos_sim(claim_emb, context_embs)[0]
                all_similarities.extend(similarities.cpu().numpy())

            # Get max similarity across all context sentences
            return float(max(all_similarities)) if all_similarities else 0.0

        except Exception as e:
            print(f"\n⚠ Semantic similarity error: {str(e)[:100]}")
            return 0.5  # Fallback score

    def nli_verify(self, claim: str, context: str) -> float:
        # Extract individual claims from the text
        claims = self._extract_claims(claim)
        if not claims:
            return 0.5  # Neutral if no claims

        scores = []
        for single_claim in claims:
            try:
                # Truncate to prevent CUDA errors (BART-MNLI max: 1024 tokens)
                # Approximate: 1 token ≈ 4 characters
                max_chars = 3000

                truncated_context = context[:max_chars] if len(context) > max_chars else context
                truncated_claim = single_claim[:500] if len(single_claim) > 500 else single_claim

                # NLI: Does context entail this claim?
                input_text = f"{truncated_context} [SEP] {truncated_claim}"

                # Skip if still too long
                if len(input_text) > max_chars:
                    scores.append(0.5)  # Neutral for oversized inputs
                    continue

                result = self.nli_model(input_text, truncation=True, max_length=1024)[0]

                # Convert label to score
                label = result['label'].upper()
                confidence = result['score']

                if 'ENTAIL' in label:
                    scores.append(confidence)
                elif 'CONTRADICT' in label:
                    scores.append(0.0)  # Clear hallucination
                else:  # NEUTRAL
                    scores.append(0.5)

            except Exception as e:
                # Handle any CUDA or processing errors gracefully
                print(f"\n⚠ NLI error (using fallback score): {str(e)[:100]}")
                scores.append(0.5)  # Neutral fallback

        return np.mean(scores) if scores else 0.5

    def cross_encoder_score(self, claim: str, context: str) -> float:
        try:
            # Cross-encoders also have token limits - truncate safely
            max_length = 2000
            claim = claim[:max_length] if len(claim) > max_length else claim
            context = context[:max_length] if len(context) > max_length else context

            # Cross-encoder computes relevance in one pass
            score = self.cross_encoder.predict([[context, claim]])[0]

            # Normalize to 0-1 range (cross-encoder outputs vary)
            normalized_score = 1 / (1 + np.exp(-score))  # Sigmoid
            return float(normalized_score)

        except Exception as e:
            print(f"\n⚠ Cross-encoder error: {str(e)[:100]}")
            return 0.5  # Fallback score

    def kg_verify(self, claim: str) -> float:
        return self.kg_verifier.verify(claim)

    def is_factual_claim(self, text: str) -> bool:
        factual_indicators = [
            r'\d{4}',  # Years
            r'\d+%',   # Percentages
            r'\d+',    # Numbers
            r'founded|established|born|died|invented|discovered',
            r'CEO|president|director|founder',
            r'located in|capital of|born in'
        ]

        for pattern in factual_indicators:
            if re.search(pattern, text, re.IGNORECASE):
                return True
        return False

    def _extract_claims(self, text: str) -> List[str]:
        sentences = sent_tokenize(text)
        # Filter out questions and very short sentences
        claims = [
            s for s in sentences
            if len(s.split()) > 5 and not s.strip().endswith('?')
        ]
        return claims


class MockKnowledgeGraphVerifier:

    def verify(self, claim: str) -> float:

        return 0.8



# EVALUATION PIPELINE FOR THE DATASET

def evaluate_dataset_with_sota(
    df: pd.DataFrame,
    batch_size: int = 32,
    checkpoint_every: int = 1000,
    resume_from: Optional[str] = None
) -> pd.DataFrame:

    detector = StateOfTheArtHallucinationDetector(use_gpu=True)

    # Resume from checkpoint if provided
    start_idx = 0
    results = []

    if resume_from and os.path.exists(resume_from):
        print(f" Resuming from checkpoint: {resume_from}")
        checkpoint_df = pd.read_csv(resume_from)
        results = checkpoint_df.to_dict('records')
        start_idx = len(results)
        print(f"✓ Loaded {start_idx} processed samples\n")

    print(f"\nEvaluating {len(df) - start_idx} samples with SOTA detector...")
    print(f"Batch size: {batch_size} | Checkpoint every: {checkpoint_every} samples\n")

    # Process in batches for efficiency
    for batch_start in tqdm(range(start_idx, len(df), batch_size)):
        batch_end = min(batch_start + batch_size, len(df))
        batch_df = df.iloc[batch_start:batch_end]

        for idx, row in batch_df.iterrows():
            try:
                content = str(row['content'])
                context = str(row.get('content_clean', ''))

                # If no context, use content itself as reference
                if not context or context == 'nan':
                    context = content

                # Detect hallucination
                result = detector.detect_hallucination(content, context)

                results.append({
                    'id': row['id'],
                    'model_name': row['model_name'],
                    'final_score': result['final_score'],
                    'is_hallucination': result['is_hallucination'],
                    'confidence': result['confidence'],
                    'semantic_score': result['breakdown']['semantic'],
                    'nli_score': result['breakdown']['nli'],
                    'cross_encoder_score': result['breakdown']['cross_encoder'],
                    'kg_score': result['breakdown']['knowledge_graph']
                })

            except Exception as e:
                print(f"\n⚠ Error processing row {idx}: {str(e)}")
                results.append({
                    'id': row['id'],
                    'model_name': row['model_name'],
                    'final_score': 0.0,
                    'is_hallucination': True,
                    'confidence': 0.0,
                    'semantic_score': 0.0,
                    'nli_score': 0.0,
                    'cross_encoder_score': 0.0,
                    'kg_score': 0.0
                })

        # Save checkpoint
        if (batch_end % checkpoint_every == 0 or batch_end == len(df)):
            checkpoint_file = f'sota_checkpoint_{batch_end}.csv'
            pd.DataFrame(results).to_csv(checkpoint_file, index=False)
            print(f"\n Checkpoint saved: {checkpoint_file}")

            # Clear GPU cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()

    return pd.DataFrame(results)



# ANALYSIS AND REPORTING

def analyze_sota_results(results_df: pd.DataFrame):


    print("STATE-OF-THE-ART HALLUCINATION DETECTION RESULTS")


    print("\n OVERALL STATISTICS")

    print(f"Total Samples: {len(results_df)}")
    print(f"Hallucinations Detected: {results_df['is_hallucination'].sum()} "
          f"({results_df['is_hallucination'].mean()*100:.1f}%)")
    print(f"Average Final Score: {results_df['final_score'].mean():.4f}")
    print(f"Average Confidence: {results_df['confidence'].mean():.4f}")

    # Component Scores
    print("\n COMPONENT SCORES (Average)")

    print(f"Semantic Similarity:  {results_df['semantic_score'].mean():.4f}")
    print(f"NLI Verification:     {results_df['nli_score'].mean():.4f}")
    print(f"Cross-Encoder:        {results_df['cross_encoder_score'].mean():.4f}")
    print(f"Knowledge Graph:      {results_df['kg_score'].mean():.4f}")

    # Per-Model Analysis
    print("\n PER-MODEL ANALYSIS")

    for model in results_df['model_name'].unique():
        model_data = results_df[results_df['model_name'] == model]
        halluc_rate = model_data['is_hallucination'].mean() * 100
        avg_score = model_data['final_score'].mean()

        print(f"\n{model.upper()}:")
        print(f"  Hallucination Rate: {halluc_rate:.1f}%")
        print(f"  Average Score:      {avg_score:.4f}")
        print(f"  Semantic:           {model_data['semantic_score'].mean():.4f}")
        print(f"  NLI:                {model_data['nli_score'].mean():.4f}")
        print(f"  Cross-Encoder:      {model_data['cross_encoder_score'].mean():.4f}")

    # Model Rankings
    print("\n MODEL RANKINGS (by Hallucination Rate - lower is better)")

    model_halluc_rates = results_df.groupby('model_name')['is_hallucination'].mean().sort_values()
    for rank, (model, rate) in enumerate(model_halluc_rates.items(), 1):
        print(f"{rank}. {model:15s}: {rate*100:5.1f}% hallucinations")

    return model_halluc_rates



# MAIN EXECUTION

def main(
    checkpoint_file: Optional[str] = None,
    batch_size: int = 32,
    checkpoint_every: int = 1000
):

    print(" State-of-the-Art Hallucination Detection Pipeline (GPU Accelerated)")


    # GPU Info
    if torch.cuda.is_available():
        print(f"\n GPU: {torch.cuda.get_device_name(0)}")
        print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        print(f"   CUDA Version: {torch.version.cuda}\n")

    # Load your dataset
    print(" Loading dataset...")
    df = pd.read_csv('combined_dataset.csv')
    print(f"✓ Loaded {len(df)} samples from {df['model_name'].nunique()} models")

    # Estimate processing time
    samples_per_sec = 2.5 if torch.cuda.is_available() else 0.5  # Rough estimates
    estimated_minutes = (len(df) / samples_per_sec) / 60
    print(f" Estimated time: {estimated_minutes:.1f} minutes on {'GPU' if torch.cuda.is_available() else 'CPU'}")

    # Run SOTA evaluation with GPU acceleration
    results_df = evaluate_dataset_with_sota(
        df,
        batch_size=batch_size,
        checkpoint_every=checkpoint_every,
        resume_from=checkpoint_file
    )

    # Analyze results
    model_rankings = analyze_sota_results(results_df)

    # Save final results
    output_file = 'sota_hallucination_results_final.csv'
    results_df.to_csv(output_file, index=False)
    print(f"\n Final results saved to: {output_file}")

    # GPU cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\n GPU memory cleared")

    return results_df, model_rankings



if __name__ == "__main__":
    import os


    BATCH_SIZE = 32
    CHECKPOINT_EVERY = 1000  # Save every 1000 samples

    RESUME_FROM = None

    # Run the pipeline
    results_df, model_rankings = main(
        checkpoint_file=RESUME_FROM,
        batch_size=BATCH_SIZE,
        checkpoint_every=CHECKPOINT_EVERY
    )

    # Display sample results
    print("\n SAMPLE RESULTS (First 10 rows):")
    print(results_df.head(10))

    print("\n SCORE DISTRIBUTION:")
    print(results_df[['semantic_score', 'nli_score', 'cross_encoder_score', 'final_score']].describe())